In [6]:
!pip install qiskit qiskit-aer qiskit-ibm-runtime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 57.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 377.4/377.4 kB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.8/75.8 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.2/130.2 kB 12.5 MB/s eta 0:00:00


In [9]:
import numpy as np
from math import log2, ceil

from qiskit import QuantumCircuit
from qiskit_aer.primitives import EstimatorV2 as AerEstimator
from qiskit.quantum_info import SparsePauliOp


# --------- Helper: pad & normalize vectors ---------

def pad_and_normalize(vec):
    """
    Pad vec to next power-of-2 length and normalize.
    Returns (state, num_qubits).
    If vec is all zeros, returns (None, num_qubits).
    """
    v = np.asarray(vec, dtype=float)
    dim = len(v)
    if dim == 0:
        raise ValueError("Input vector has dimension 0.")

    num_qubits = ceil(log2(dim)) if dim > 0 else 0
    required_dim = 2**num_qubits

    if dim < required_dim:
        v = np.pad(v, (0, required_dim - dim), mode="constant", constant_values=0.0)

    norm = np.linalg.norm(v)
    if norm == 0:
        return None, num_qubits

    return v / norm, num_qubits


# --------- SWAP test circuit WITHOUT measurement ---------

def swap_test_circuit_from_vectors(v1, v2):
    """
    Build a SWAP test circuit from two classical vectors using amplitude encoding.
    Returns (qc, ancilla_index).

    This is your old qubit-by-qubit SWAP test, but:
    - no classical register,
    - no measurement,
    so it can be used with Estimator/NEAT.
    """
    # Pad + normalize
    v1_state_norm, nq1 = pad_and_normalize(v1)
    v2_state_norm, nq2 = pad_and_normalize(v2)

    if v1_state_norm is None or v2_state_norm is None:
        raise ValueError("One of the vectors is all zeros; cannot normalize.")

    if nq1 != nq2:
        raise ValueError("Vector dimensions (after padding) do not match.")

    num_qubits_vec = nq1
    total_qubits = 2 * num_qubits_vec + 1
    qc = QuantumCircuit(total_qubits, name="swap_test")

    # logical indices
    q1_l = list(range(num_qubits_vec))
    q2_l = list(range(num_qubits_vec, 2 * num_qubits_vec))
    ancilla = 2 * num_qubits_vec

    # State preparation (amplitude encoding)
    qc.initialize(v1_state_norm, q1_l)
    qc.initialize(v2_state_norm, q2_l)

    # SWAP-test core (same as your old code, just without measure)
    qc.h(ancilla)
    for i in range(num_qubits_vec):
        qc.cswap(ancilla, q1_l[i], q2_l[i])
    qc.h(ancilla)

    return qc, ancilla


# --------- Build an Estimator "pub" for this SWAP test ---------

def build_swap_pub_from_vectors(v1, v2):
    """
    Build a Primitive Unified Bloc (pub) suitable for EstimatorV2:
      pub = (circuit, [observable])

    Observable is Z on the ancilla qubit, encoded as a SparsePauliOp.
    """
    qc, ancilla = swap_test_circuit_from_vectors(v1, v2)
    n = qc.num_qubits

    # Pauli string with Z on ancilla, I everywhere else
    pauli_str = "I" * ancilla + "Z" + "I" * (n - ancilla - 1)
    observable = SparsePauliOp(pauli_str)

    pub = (qc, [observable])
    return pub


In [10]:
def ideal_fidelity_with_aer(v1, v2, precision=0.0):
    """
    Use qiskit_aer.primitives.EstimatorV2 to get an ideal estimate of:
      F = |⟨ψ|φ⟩|^2  (via ⟨Z_ancilla⟩)

    If precision=0.0, this is an exact statevector evaluation.
    If precision>0, Aer will sample N(expval, precision).
    """
    estimator = AerEstimator(options={"default_precision": precision})

    pub = build_swap_pub_from_vectors(v1, v2)
    job = estimator.run([pub])
    result = job.result()

    # result[0] is the PubResult; .data.evs is the array of expectation values
    # (one per observable in the pub). For us, it's a single Z on the ancilla.
    fidelity = result[0].data.evs[0]   # ⟨Z_ancilla⟩ = |⟨ψ|φ⟩|^2 for SWAP test

    # Convert to probability of measuring ancilla = 0, if you want to compare
    # directly with your old count-based implementation.
    p0 = (1 + fidelity) / 2.0

    return fidelity, p0


# --- quick smoke test ---

# if __name__ == "__main__":
    # Two simple vectors
v1 = [1, 0, 0, 0]
v2 = [1, 0, 0, 0]  # identical => fidelity ≈ 1

F, p0 = ideal_fidelity_with_aer(v1, v2)
print(f"Fidelity (ideal, Aer Estimator) = {F}")
print(f"P(ancilla=0) = {p0}")


Fidelity (ideal, Aer Estimator) = 1.0
P(ancilla=0) = 1.0


In [11]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime.debug_tools import Neat

def ideal_fidelity_with_neat(v1, v2, backend_name=None):
    """
    Same pub as before, but run through NEAT. This is essentially a
    wrapper over Aer EstimatorV2, using the IBM Runtime stack.
    """
    # Build the pub from your vectors
    pub = build_swap_pub_from_vectors(v1, v2)
    pubs = [pub]

    # Connect to IBM Quantum / Runtime
    service = QiskitRuntimeService()

    if backend_name is None:
        # pick an operational QPU (or you can choose a specific backend)
        backend = service.least_busy(operational=True, simulator=False)
    else:
        backend = service.backend(backend_name)

    analyzer = Neat(backend, noise_model=None)

    # (Optional) Cliffordize the pub for scalable classical sim
    # If your state prep uses non-Clifford gates, to_clifford() will map
    # to a nearby Clifford circuit with similar structure.
    clifford_pubs = analyzer.to_clifford(pubs)

    # Ideal (noiseless) simulation of expectation value(s)
    ideal_results = analyzer.ideal_sim(clifford_pubs, precision=0)

    # NeatResult is iterable; each entry is a NeatPubResult with .vals (array)
    fidelity = ideal_results[0].vals[0]   # single observable (ancilla Z)
    p0 = (1 + fidelity) / 2.0

    return fidelity, p0
